# OMOP Visit Occurrence Table

Transforms FHIR Encounter resources into OMOP CDM `visit_occurrence` table.

## Mapping: FHIR Encounter → OMOP Visit_Occurrence

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| visit_occurrence_id | Encounter.id | Hash to integer |
| person_id | Encounter.subject | Reference to person |
| visit_concept_id | Encounter.class | Map to OMOP visit concepts |
| visit_start_date | Encounter.period.start | Extract date |
| visit_start_datetime | Encounter.period.start | Full timestamp |
| visit_end_date | Encounter.period.end | Extract date |
| visit_end_datetime | Encounter.period.end | Full timestamp |
| visit_type_concept_id | - | 32817 (EHR) |
| provider_id | Encounter.participant | Reference to provider |
| care_site_id | Encounter.serviceProvider | Reference to care_site |
| visit_source_value | Encounter.id | Original FHIR ID |
| visit_source_concept_id | Encounter.class | Source concept |

## Visit Concept Mapping

| FHIR Encounter.class | OMOP visit_concept_id | Description |
|---------------------|----------------------|-------------|
| IMP, inpatient | 9201 | Inpatient Visit |
| AMB, outpatient | 9202 | Outpatient Visit |
| EMER, emergency | 9203 | Emergency Room Visit |
| HH, home | 581476 | Home Visit |
| VR, virtual | 5083 | Telehealth |
| Other | 0 | Unknown |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Visit Occurrence Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_visit_stmt STRING;

SET VARIABLE create_visit_stmt = "
CREATE OR REFRESH STREAMING TABLE visit_occurrence (
  -- Primary key
  visit_occurrence_id BIGINT NOT NULL COMMENT 'Unique visit identifier'
  
  -- Person reference
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  
  -- Visit classification
  ,visit_concept_id INT NOT NULL COMMENT 'Visit type: 9201=Inpatient, 9202=Outpatient, 9203=ER'
  
  -- Dates
  ,visit_start_date DATE NOT NULL COMMENT 'Visit start date'
  ,visit_start_datetime TIMESTAMP COMMENT 'Visit start datetime'
  ,visit_end_date DATE COMMENT 'Visit end date'
  ,visit_end_datetime TIMESTAMP COMMENT 'Visit end datetime'
  
  -- Type
  ,visit_type_concept_id INT NOT NULL DEFAULT 32817 COMMENT 'Type concept: 32817=EHR'
  
  -- References
  ,provider_id BIGINT COMMENT 'Reference to provider table'
  ,care_site_id BIGINT COMMENT 'Reference to care_site table'
  
  -- Source values
  ,visit_source_value STRING COMMENT 'Original FHIR Encounter ID'
  ,visit_source_concept_id INT DEFAULT 0 COMMENT 'Source concept for visit type'
  
  -- Admission/Discharge
  ,admitted_from_concept_id INT DEFAULT 0 COMMENT 'Admission source concept'
  ,admitted_from_source_value STRING COMMENT 'Admission source value'
  ,discharged_to_concept_id INT DEFAULT 0 COMMENT 'Discharge disposition concept'
  ,discharged_to_source_value STRING COMMENT 'Discharge disposition value'
  
  -- Visit linkage
  ,preceding_visit_occurrence_id BIGINT COMMENT 'Reference to prior visit'
  
  -- Lineage
  ,fhir_encounter_uuid STRING COMMENT 'Original FHIR Encounter UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Visit Occurrence table - Encounters from FHIR'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer visit_occurrence_id
  ABS(HASH(COALESCE(id::STRING, encounter_uuid))) AS visit_occurrence_id
  
  -- Person reference (hash the patient reference to match person table)
  ,ABS(HASH(
    COALESCE(
      REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
      subject:reference::STRING
    )
  )) AS person_id
  
  -- Map encounter class to visit_concept_id
  ,CASE 
    WHEN UPPER(class:code::STRING) IN ('IMP', 'INPATIENT', 'INP') THEN 9201
    WHEN UPPER(class:code::STRING) IN ('AMB', 'OUTPATIENT', 'OUT') THEN 9202
    WHEN UPPER(class:code::STRING) IN ('EMER', 'EMERGENCY', 'ER') THEN 9203
    WHEN UPPER(class:code::STRING) IN ('HH', 'HOME') THEN 581476
    WHEN UPPER(class:code::STRING) IN ('VR', 'VIRTUAL', 'TELEHEALTH') THEN 5083
    WHEN UPPER(class:code::STRING) = 'OBSENC' THEN 9201  -- Observation encounter treated as inpatient
    ELSE 0
  END AS visit_concept_id
  
  -- Dates from period
  ,CAST(TRY_CAST(period:start::STRING AS TIMESTAMP) AS DATE) AS visit_start_date
  ,TRY_CAST(period:start::STRING AS TIMESTAMP) AS visit_start_datetime
  ,COALESCE(
    CAST(TRY_CAST(period:end::STRING AS TIMESTAMP) AS DATE),
    CAST(TRY_CAST(period:start::STRING AS TIMESTAMP) AS DATE)  -- Default end = start for outpatient
  ) AS visit_end_date
  ,TRY_CAST(period:end::STRING AS TIMESTAMP) AS visit_end_datetime
  
  -- Type concept (EHR)
  ,32817 AS visit_type_concept_id
  
  -- References (to be populated when provider/care_site tables exist)
  ,NULL AS provider_id
  ,NULL AS care_site_id
  
  -- Source values
  ,id::STRING AS visit_source_value
  ,0 AS visit_source_concept_id
  
  -- Admission/Discharge (from hospitalization if available)
  ,0 AS admitted_from_concept_id
  ,hospitalization:admitSource:coding[0]:code::STRING AS admitted_from_source_value
  ,0 AS discharged_to_concept_id
  ,hospitalization:dischargeDisposition:coding[0]:code::STRING AS discharged_to_source_value
  
  -- No preceding visit linkage in initial load
  ,NULL AS preceding_visit_occurrence_id
  
  -- Lineage
  ,encounter_uuid AS fhir_encounter_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".encounter)
WHERE id IS NOT NULL
  AND status::STRING != 'cancelled'
";

SELECT create_visit_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_visit_stmt;

In [ ]:
-- Verify visit_occurrence table
SELECT 
  visit_occurrence_id,
  person_id,
  visit_concept_id,
  visit_start_date,
  visit_end_date,
  visit_source_value
FROM visit_occurrence
LIMIT 10;

In [ ]:
-- Visit type distribution
SELECT 
  CASE visit_concept_id
    WHEN 9201 THEN 'Inpatient'
    WHEN 9202 THEN 'Outpatient'
    WHEN 9203 THEN 'Emergency'
    WHEN 581476 THEN 'Home'
    WHEN 5083 THEN 'Telehealth'
    ELSE 'Other/Unknown'
  END AS visit_type,
  COUNT(*) AS count
FROM visit_occurrence
GROUP BY visit_concept_id
ORDER BY count DESC;